## Import libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import date
from pathlib import Path

# Files

## Files path

In [2]:
DATA_RRHH_RAW_FILES = [
    '../../Data/raw/raw_data_25052026.csv',
    '../../Data/raw/raw_data_01062026.csv',
    '../../Data/raw/raw_data_08062026.csv'
]

DATA_RRHH_RAW_FILE = DATA_RRHH_RAW_FILES[-1]
DATA_RRHH_OLD_RAW_FILE = DATA_RRHH_RAW_FILES[-2]

DATA_OUTPUT_FILE = '../../Data/clean/clean_data_08062026.csv'


## Read files

In [3]:
df_RRHH = pd.read_csv(DATA_RRHH_RAW_FILE)
df_RRHH.sample(5)

,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Disciplinary_failure,Education,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,Absenteeism_hours
106,20,19,3,3,2,260,50,11,36,"343,253",...,0,1,4,1,0,0,65,168,23,8
285,1,23,8,5,1,235,11,14,37,"205,917",...,0,3,1,0,0,1,88,172,29,4
954,246,11,7,4,1,179,51,18,38,"239,554",...,0,1,0,1,0,0,89,170,31,1
436,3,27,3,3,3,179,51,18,38,"222,196",...,0,1,0,1,0,0,89,170,31,3
822,119,7,7,2,1,268,11,8,33,"230,290",...,0,2,0,0,0,0,79,178,25,8


In [4]:
df_RRHH_old= pd.read_csv(DATA_RRHH_OLD_RAW_FILE)
df_RRHH_old

,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Disciplinary_failure,Education,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,Absenteeism_hours
0,14,11,11,2,4,155,12,14,34,"284,031",...,0,1,2,1,0,0,95,196,25,120
1,36,13,4,4,3,118,13,18,50,"239,409",...,0,1,1,1,0,0,98,178,31,120
2,9,6,7,3,1,228,14,16,58,"264,604",...,0,1,2,0,0,1,65,172,22,120
3,28,9,7,3,1,225,26,9,28,"230,290",...,0,1,1,0,0,2,69,169,24,112
4,9,12,3,3,2,228,14,16,58,"222,196",...,0,1,2,0,0,1,65,172,22,112
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
840,14,11,11,2,4,155,12,14,34,"284,031",...,0,1,2,1,0,0,95,196,25,120
841,36,13,4,4,3,118,13,18,50,"239,409",...,0,1,1,1,0,0,98,178,31,120
842,9,6,7,3,1,228,14,16,58,"264,604",...,0,1,2,0,0,1,65,172,22,120
843,28,9,7,3,1,225,26,9,28,"230,290",...,0,1,1,0,0,2,69,169,24,112


## Check de Registros

In [5]:
def control_registros(df):

    fecha_actual = date.today()

    total_registros = df.shape[0]
    ID_unicos = df['ID'].nunique()
    ID_extra = total_registros - ID_unicos
    ID_duplicados_exactos = df[df.duplicated(keep=False)].drop_duplicates().shape[0]


    print('--------------------\nCONTROL DE REGISTROS\n--------------------')
    print(f'Fecha: {fecha_actual}\n--------------------')
    print(f'TOTAL REGISTROS: {total_registros}')
    print(f'ID ÚNICOS: {ID_unicos}')
    print(f'ID EXTRAS: {ID_extra}')
    print(f'ID DUPLICADOS EXACTOS: {ID_duplicados_exactos}')

In [ ]:
# En viamos el print y esperamos confirmación de Verónica
control_registros(df_RRHH)

--------------------
CONTROL DE REGISTROS
--------------------
Fecha: 2026-06-10
--------------------
TOTAL REGISTROS: 1110
ID ÚNICOS: 386
ID EXTRAS: 724
ID DUPLICADOS EXACTOS: 41


## Añadir columna Importacion

In [7]:
def get_import_date(file_path):
    date_text = Path(file_path).stem.split('_')[-1]
    return pd.to_datetime(date_text, format='%d%m%Y').strftime('%Y-%m-%d')

IMPORTACION_ACTUAL = get_import_date(DATA_RRHH_RAW_FILE)
IMPORTACION_ANTERIOR = get_import_date(DATA_RRHH_OLD_RAW_FILE)


In [8]:
# Los raw son acumulativos: cada nueva entrega conserva las filas anteriores
# y añade nuevos registros al final. La fecha de importación se asigna según
# el límite de filas alcanzado por cada entrega, aunque una fila histórica
# haya sido corregida posteriormente.
fechas_importacion = []
limite_anterior = 0
ids_anteriores = set()
resumen_importaciones = []

for raw_file in DATA_RRHH_RAW_FILES:
    raw_ids = pd.read_csv(raw_file, usecols=['ID'])
    total_hasta_entrega = raw_ids.shape[0]
    ids_entrega = set(raw_ids['ID'])
    nuevos_registros = total_hasta_entrega - limite_anterior

    if nuevos_registros < 0:
        raise ValueError(
            f'El raw {raw_file} tiene menos registros que la entrega anterior.'
        )

    fecha_importacion = get_import_date(raw_file)
    fechas_importacion.extend([fecha_importacion] * nuevos_registros)
    resumen_importaciones.append({
        'importacion': fecha_importacion,
        'registros_nuevos': nuevos_registros,
        'IDs_nuevos': len(ids_entrega - ids_anteriores),
        'registros_acumulados': total_hasta_entrega,
        'IDs_acumulados': len(ids_entrega)
    })
    limite_anterior = total_hasta_entrega
    ids_anteriores = ids_entrega

if len(fechas_importacion) != len(df_RRHH):
    raise ValueError(
        'El último raw no coincide con el histórico acumulativo declarado.'
    )

df_RRHH['importacion'] = fechas_importacion

print('Registros cargados:', df_RRHH.shape[0])
print('Columnas:', df_RRHH.shape[1])
display(pd.DataFrame(resumen_importaciones))

df_RRHH.sample(5)


Registros cargados: 1110
Columnas: 22


,importacion,registros_nuevos,IDs_nuevos,registros_acumulados,IDs_acumulados
0,2026-05-25,740,36,740,36
1,2026-06-01,105,100,845,136
2,2026-06-08,265,250,1110,386


,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Education,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,Absenteeism_hours,importacion
188,3,28,4,4,3,179,51,18,38,"239,409",...,1,0,1,0,0,89,170,31,8,2026-05-25
383,3,27,2,4,2,179,51,18,38,"251,818",...,1,0,1,0,0,89,170,31,3,2026-05-25
247,1,22,4,6,3,235,11,14,37,"246,288",...,3,1,0,0,1,88,172,29,8,2026-05-25
267,9,6,7,2,1,228,14,16,58,"264,604",...,1,2,0,0,1,65,172,22,8,2026-05-25
1035,327,23,10,3,4,179,51,18,38,"253,465",...,1,0,1,0,0,89,170,31,3,2026-06-08


# Data cleaning

In [9]:
df_RRHH.describe().T #Traspone columna

,count,mean,std,min,25%,50%,75%,max
ID,1110.0,79.075676,106.713785,1.0,13.25,28.0,108.75,386.0
Reason_absence,1110.0,18.977477,8.479564,0.0,13.00,23.0,26.00,28.0
Month_absence,1110.0,6.381982,3.414555,0.0,3.00,6.0,9.00,12.0
Day_week,1110.0,3.902703,1.434312,2.0,3.00,4.0,5.00,6.0
Seasons,1110.0,2.536937,1.116313,1.0,2.00,3.0,4.00,4.0
Transportation_expense,1110.0,221.316216,66.585525,118.0,179.00,225.0,260.00,388.0
Distance_Residence_Work,1110.0,29.504505,14.798237,5.0,16.00,26.0,50.00,52.0
Service_time,1110.0,12.652252,4.369103,1.0,9.00,13.0,16.00,29.0
Age,1110.0,36.639640,6.657935,27.0,31.00,37.0,40.00,58.0
Hit_target,1110.0,94.634234,3.746173,81.0,93.00,95.0,97.00,100.0


In [10]:
df_RRHH.info()

<class 'pandas.DataFrame'>
RangeIndex: 1110 entries, 0 to 1109
Data columns (total 22 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   ID                       1110 non-null   int64
 1   Reason_absence           1110 non-null   int64
 2   Month_absence            1110 non-null   int64
 3   Day_week                 1110 non-null   int64
 4   Seasons                  1110 non-null   int64
 5   Transportation_expense   1110 non-null   int64
 6   Distance_Residence_Work  1110 non-null   int64
 7   Service_time             1110 non-null   int64
 8   Age                      1110 non-null   int64
 9   Work_load_Average_day    1110 non-null   str  
 10  Hit_target               1110 non-null   int64
 11  Disciplinary_failure     1110 non-null   int64
 12  Education                1110 non-null   int64
 13  Son                      1110 non-null   int64
 14  Social_drinker           1110 non-null   int64
 15  Social_smoker  

In [11]:
df_RRHH["importacion"] = pd.to_datetime(df_RRHH["importacion"], format="%Y-%m-%d")
df_RRHH.sample(5)

,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Education,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,Absenteeism_hours,importacion
213,13,23,10,3,4,369,17,12,31,"284,853",...,1,3,1,0,0,70,169,25,8,2026-05-25
441,29,19,5,4,3,225,15,15,41,"237,656",...,4,2,1,0,2,94,182,28,3,2026-05-25
781,78,22,3,6,3,289,36,13,33,"244,387",...,1,2,1,0,1,90,172,30,8,2026-06-01
62,6,22,7,3,1,189,29,13,33,"264,604",...,1,2,0,0,2,69,167,25,16,2026-05-25
873,165,28,8,3,1,246,25,16,41,"249,797",...,1,0,1,0,0,67,170,23,4,2026-06-08


## Duplicated consistency - "Importacion" excluded

In [12]:
#DF backup del original
df_rrhh_copy = df_RRHH.drop(columns=["importacion"]).copy()
df_rrhh_copy

,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Disciplinary_failure,Education,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,Absenteeism_hours
0,14,11,11,2,4,155,12,14,34,"284,031",...,0,1,2,1,0,0,95,196,25,120
1,36,13,4,4,3,118,13,18,50,"239,409",...,0,1,1,1,0,0,98,178,31,120
2,9,6,7,3,1,228,14,16,58,"264,604",...,0,1,2,0,0,1,65,172,22,120
3,28,9,7,3,1,225,26,9,28,"230,290",...,0,1,1,0,0,2,69,169,24,112
4,9,12,3,3,2,228,14,16,58,"222,196",...,0,1,2,0,0,1,65,172,22,112
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1105,22,1,10,4,4,179,26,9,30,"265,017",...,0,3,0,0,0,0,56,171,19,64
1106,26,19,5,6,3,300,26,13,43,"237,656",...,0,1,2,1,1,1,77,175,25,64
1107,34,19,12,3,4,118,10,10,37,"261,306",...,0,1,0,0,0,0,83,172,28,56
1108,20,19,4,6,3,260,50,11,36,"326,452",...,0,1,4,1,0,0,65,168,23,56


In [13]:
ID_distintos_con_duplicados = df_rrhh_copy.loc[df_rrhh_copy.duplicated(keep=False), 'ID'].nunique()
ID_distintos_con_duplicados_old = df_RRHH_old.loc[df_RRHH_old.duplicated(keep=False), 'ID'].nunique()
print(f'Esta semana los id con duplicados son: {ID_distintos_con_duplicados} y la semana pasada eran: {ID_distintos_con_duplicados_old}')

Esta semana los id con duplicados son: 16 y la semana pasada eran: 12


In [14]:
grupos_filas_duplicadas= df_rrhh_copy[df_rrhh_copy.duplicated(keep=False)].drop_duplicates().shape[0]
grupos_filas_duplicadas_old= df_RRHH_old[df_RRHH_old.duplicated(keep=False)].drop_duplicates().shape[0]
print(f'Esta semana los grupos de filas duplicadas son: {grupos_filas_duplicadas} y la semana pasada eran: {grupos_filas_duplicadas_old}')

Esta semana los grupos de filas duplicadas son: 41 y la semana pasada eran: 31


In [15]:
#Duplicados actuales y su conteo
df_rrhh_copy[df_rrhh_copy.duplicated(keep=False)].value_counts().reset_index(name='n_duplicados')

,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Education,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,Absenteeism_hours,n_duplicados
0,3,27,2,4,2,179,51,18,38,"251,818",...,1,0,1,0,0,89,170,31,3,4
1,22,27,4,6,3,179,26,9,30,"246,288",...,3,0,0,0,0,56,171,19,2,4
2,14,11,11,2,4,155,12,14,34,"284,031",...,1,2,1,0,0,95,196,25,120,3
3,36,13,4,4,3,118,13,18,50,"239,409",...,1,1,1,0,0,98,178,31,120,3
4,9,6,7,3,1,228,14,16,58,"264,604",...,1,2,0,0,1,65,172,22,120,3
5,28,9,7,3,1,225,26,9,28,"230,290",...,1,1,0,0,2,69,169,24,112,3
6,9,12,3,3,2,228,14,16,58,"222,196",...,1,2,0,0,1,65,172,22,112,3
7,3,27,2,6,2,179,51,18,38,"251,818",...,1,0,1,0,0,89,170,31,3,3
8,3,27,3,5,2,179,51,18,38,"222,196",...,1,0,1,0,0,89,170,31,3,3
9,3,27,2,4,2,179,51,18,38,"264,249",...,1,0,1,0,0,89,170,31,2,3


Comparison between old and new dataset

In [16]:
comparison = pd.DataFrame({
    "old": df_RRHH_old.loc[df_RRHH_old.duplicated(keep=False), 'ID'].value_counts(),
    "new": df_rrhh_copy.loc[df_rrhh_copy.duplicated(keep=False), 'ID'].value_counts()
}).fillna(0).astype(int)

comparison["diff"] = comparison["new"] - comparison["old"]

comparison.sort_values("diff", ascending=False)

,old,new,diff
ID,,,
14,2,7,5
34,10,14,4
36,2,5,3
9,4,6,2
11,0,2,2
13,0,2,2
20,0,2,2
22,8,10,2
26,0,2,2


In [17]:
comparison = pd.DataFrame({
    "old": df_RRHH_old.loc[df_RRHH_old.duplicated(keep=False), 'ID'].value_counts(),
    "new": df_rrhh_copy.loc[df_rrhh_copy.duplicated(keep=False), 'ID'].value_counts()
}).fillna(0).astype(int)

comparison["diff"] = comparison["new"] - comparison["old"]

comparison = comparison[comparison["diff"] != 0]
comparison.sort_values("diff", ascending=False)

,old,new,diff
ID,,,
14,2,7,5
34,10,14,4
36,2,5,3
9,4,6,2
11,0,2,2
13,0,2,2
20,0,2,2
22,8,10,2
26,0,2,2


- Los duplicados se mantienen porque el dataset no contiene una fecha ni un identificador único del evento de ausencia.
- Dos ausencias distintas pueden compartir exactamente los mismos valores.
- El control anterior permite detectar variaciones semanales sin eliminar registros de forma automática.


In [18]:
df_rrhh_copy[df_rrhh_copy['ID'] == 28][df_rrhh_copy[df_rrhh_copy['ID'] == 28].duplicated(keep=False)]

,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Disciplinary_failure,Education,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,Absenteeism_hours
3,28,9,7,3,1,225,26,9,28,"230,290",...,0,1,1,0,0,2,69,169,24,112
410,28,23,12,4,4,225,26,9,28,"280,549",...,0,1,1,0,0,2,69,169,24,3
412,28,23,12,4,4,225,26,9,28,"280,549",...,0,1,1,0,0,2,69,169,24,3
616,28,23,11,4,4,225,26,9,28,"306,345",...,0,1,1,0,0,2,69,169,24,1
617,28,23,11,4,4,225,26,9,28,"306,345",...,0,1,1,0,0,2,69,169,24,1
843,28,9,7,3,1,225,26,9,28,"230,290",...,0,1,1,0,0,2,69,169,24,112
1098,28,9,7,3,1,225,26,9,28,"230,290",...,0,1,1,0,0,2,69,169,24,112


## Change Work_load_Average_day to float (, to .)

In [19]:
df_RRHH['Work_load_Average_day'] = df_RRHH['Work_load_Average_day'].str.replace(',', '.', regex=False).astype(float)

In [20]:
df_RRHH.info()

<class 'pandas.DataFrame'>
RangeIndex: 1110 entries, 0 to 1109
Data columns (total 22 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   ID                       1110 non-null   int64         
 1   Reason_absence           1110 non-null   int64         
 2   Month_absence            1110 non-null   int64         
 3   Day_week                 1110 non-null   int64         
 4   Seasons                  1110 non-null   int64         
 5   Transportation_expense   1110 non-null   int64         
 6   Distance_Residence_Work  1110 non-null   int64         
 7   Service_time             1110 non-null   int64         
 8   Age                      1110 non-null   int64         
 9   Work_load_Average_day    1110 non-null   float64       
 10  Hit_target               1110 non-null   int64         
 11  Disciplinary_failure     1110 non-null   int64         
 12  Education                1110 non-null   int6

## Month_absence 0 - to nan

Check df

In [21]:
df_RRHH[df_RRHH['Month_absence'] == 0]

,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Education,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,Absenteeism_hours,importacion
737,4,0,0,3,1,118,14,13,40,271.219,...,1,1,1,0,8,98,170,34,0,2026-05-25
738,8,0,0,4,2,231,35,14,39,271.219,...,1,2,1,0,2,100,170,35,0,2026-05-25
739,35,0,0,6,3,179,45,14,53,271.219,...,1,1,0,0,1,77,175,25,0,2026-05-25


In [22]:
df_RRHH[(df_RRHH['Reason_absence'] == 0) & (df_RRHH['Disciplinary_failure'] == 0)]

,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Education,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,Absenteeism_hours,importacion
737,4,0,0,3,1,118,14,13,40,271.219,...,1,1,1,0,8,98,170,34,0,2026-05-25
738,8,0,0,4,2,231,35,14,39,271.219,...,1,2,1,0,2,100,170,35,0,2026-05-25
739,35,0,0,6,3,179,45,14,53,271.219,...,1,1,0,0,1,77,175,25,0,2026-05-25


Change

In [23]:
df_RRHH['Month_absence'] = df_RRHH['Month_absence'].replace(0, np.nan)

Check change correct

In [24]:
df_RRHH['Month_absence'].unique()

array([11.,  4.,  7.,  3.,  6., 12., 10.,  5.,  8.,  9.,  1.,  2., nan])

In [25]:
df_RRHH.info()

<class 'pandas.DataFrame'>
RangeIndex: 1110 entries, 0 to 1109
Data columns (total 22 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   ID                       1110 non-null   int64         
 1   Reason_absence           1110 non-null   int64         
 2   Month_absence            1107 non-null   float64       
 3   Day_week                 1110 non-null   int64         
 4   Seasons                  1110 non-null   int64         
 5   Transportation_expense   1110 non-null   int64         
 6   Distance_Residence_Work  1110 non-null   int64         
 7   Service_time             1110 non-null   int64         
 8   Age                      1110 non-null   int64         
 9   Work_load_Average_day    1110 non-null   float64       
 10  Hit_target               1110 non-null   int64         
 11  Disciplinary_failure     1110 non-null   int64         
 12  Education                1110 non-null   int6

## BMI - changed due to inconsistencies and add one decimal (medically relevant)

In [26]:
df_bmi = df_RRHH[['Weight','Height', 'Body_mass_index']].copy()
df_bmi

,Weight,Height,Body_mass_index
0,95,196,25
1,98,178,31
2,65,172,22
3,69,169,24
4,65,172,22
...,...,...,...
1105,56,171,19
1106,77,175,25
1107,83,172,28
1108,65,168,23


Since we have both weight and height, calculate BMI

In [27]:
df_bmi['BMI_calculated'] = (
    df_bmi['Weight'] / ((df_bmi['Height'] / 100) ** 2)
)

In [28]:
df_bmi['BMI_calculated_round'] = (
    df_bmi['Weight'] / ((df_bmi['Height'] / 100) ** 2)
).round().astype(int)

In [29]:
df_bmi['BMI_calculated_round_1'] = (
    df_bmi['Weight'] / ((df_bmi['Height'] / 100) ** 2)
).round(1)

In [30]:
df_bmi[df_bmi['BMI_calculated_round'] != df_bmi['Body_mass_index']] #filas donde las columnas no coinciden

,Weight,Height,Body_mass_index,BMI_calculated,BMI_calculated_round,BMI_calculated_round_1
50,88,172,29,29.745809,30,29.7
64,88,172,29,29.745809,30,29.7
89,88,172,29,29.745809,30,29.7
145,88,172,29,29.745809,30,29.7
178,88,172,29,29.745809,30,29.7
182,88,172,29,29.745809,30,29.7
193,88,172,29,29.745809,30,29.7
196,88,172,29,29.745809,30,29.7
218,75,178,25,23.671254,24,23.7
222,75,178,25,23.671254,24,23.7


Change (Insert new column calculated next to previous column)

In [31]:
df_RRHH.insert(20,'BMI_calculated',(
    df_RRHH['Weight'] / ((df_RRHH['Height'] / 100) ** 2)
).round(1))

Check change

In [32]:
df_RRHH[['ID','BMI_calculated','Body_mass_index']]

,ID,BMI_calculated,Body_mass_index
0,14,24.7,25
1,36,30.9,31
2,9,22.0,22
3,28,24.2,24
4,9,22.0,22
...,...,...,...
1105,22,19.2,19
1106,26,25.1,25
1107,34,28.1,28
1108,20,23.0,23


In [33]:
df_RRHH.info()

<class 'pandas.DataFrame'>
RangeIndex: 1110 entries, 0 to 1109
Data columns (total 23 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   ID                       1110 non-null   int64         
 1   Reason_absence           1110 non-null   int64         
 2   Month_absence            1107 non-null   float64       
 3   Day_week                 1110 non-null   int64         
 4   Seasons                  1110 non-null   int64         
 5   Transportation_expense   1110 non-null   int64         
 6   Distance_Residence_Work  1110 non-null   int64         
 7   Service_time             1110 non-null   int64         
 8   Age                      1110 non-null   int64         
 9   Work_load_Average_day    1110 non-null   float64       
 10  Hit_target               1110 non-null   int64         
 11  Disciplinary_failure     1110 non-null   int64         
 12  Education                1110 non-null   int6

## Delete row of ID 29 with different demographics

In [34]:
df_RRHH[df_RRHH['ID'] == 29]

,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,BMI_calculated,Absenteeism_hours,importacion
252,29,14,5.0,5,3,225,15,15,41,237.656,...,2,1,0,2,94,182,28,28.4,8,2026-05-25
253,29,22,5.0,6,3,225,15,15,41,237.656,...,2,1,0,2,94,182,28,28.4,8,2026-05-25
441,29,19,5.0,4,3,225,15,15,41,237.656,...,2,1,0,2,94,182,28,28.4,3,2026-05-25
555,29,28,2.0,6,2,225,15,15,41,264.249,...,2,1,0,2,94,182,28,28.4,2,2026-05-25
698,29,0,9.0,2,4,225,26,9,28,241.476,...,1,0,0,2,69,169,24,24.2,0,2026-05-25


In [35]:
df_RRHH[(df_RRHH['Age'] == 28) & (df_RRHH['Distance_Residence_Work'] == 26) & (df_RRHH['Service_time'] == 9)
        & (df_RRHH['Weight'] == 69) & (df_RRHH['Height'] == 169) & (df_RRHH['Body_mass_index'] == 24) & (df_RRHH['Son'] == 1)]

,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,BMI_calculated,Absenteeism_hours,importacion
3,28,9,7.0,3,1,225,26,9,28,230.290,...,1,0,0,2,69,169,24,24.2,112,2026-05-25
47,28,11,3.0,4,3,225,26,9,28,343.253,...,1,0,0,2,69,169,24,24.2,16,2026-05-25
115,28,11,3.0,2,3,225,26,9,28,343.253,...,1,0,0,2,69,169,24,24.2,8,2026-05-25
116,28,11,3.0,3,3,225,26,9,28,343.253,...,1,0,0,2,69,169,24,24.2,8,2026-05-25
123,28,19,5.0,3,3,225,26,9,28,378.884,...,1,0,0,2,69,169,24,24.2,8,2026-05-25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1061,353,9,7.0,3,1,225,26,9,28,230.290,...,1,0,0,2,69,169,24,24.2,112,2026-06-08
1068,360,19,4.0,5,3,225,26,9,28,246.288,...,1,0,0,2,69,169,24,24.2,8,2026-06-08
1071,363,28,2.0,5,2,225,26,9,28,264.249,...,1,0,0,2,69,169,24,24.2,3,2026-06-08
1075,367,7,3.0,2,2,225,26,9,28,222.196,...,1,0,0,2,69,169,24,24.2,8,2026-06-08


Se elimina el único registro del ID 29 con características demográficas incompatibles. Antes de eliminarlo, el código verifica que el índice y los valores esperados no hayan cambiado.


In [36]:
registro_erroneo_index = 698
criterios_registro_erroneo = {
    'ID': 29,
    'Age': 28,
    'Distance_Residence_Work': 26,
    'Service_time': 9,
    'Weight': 69,
    'Height': 169,
    'Body_mass_index': 24,
    'Son': 1
}

assert registro_erroneo_index in df_RRHH.index, (
    f'No existe el índice previsto {registro_erroneo_index}.'
)
assert all(
    df_RRHH.loc[registro_erroneo_index, columna] == valor
    for columna, valor in criterios_registro_erroneo.items()
), 'El índice 698 ya no corresponde al registro erróneo esperado.'

df_RRHH = df_RRHH.drop(index=registro_erroneo_index)


Check

In [37]:
df_RRHH[df_RRHH['ID'] == 29]

,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,BMI_calculated,Absenteeism_hours,importacion
252,29,14,5.0,5,3,225,15,15,41,237.656,...,2,1,0,2,94,182,28,28.4,8,2026-05-25
253,29,22,5.0,6,3,225,15,15,41,237.656,...,2,1,0,2,94,182,28,28.4,8,2026-05-25
441,29,19,5.0,4,3,225,15,15,41,237.656,...,2,1,0,2,94,182,28,28.4,3,2026-05-25
555,29,28,2.0,6,2,225,15,15,41,264.249,...,2,1,0,2,94,182,28,28.4,2,2026-05-25


## Resumen semanal de registros e IDs incorporados


In [38]:
df_RRHH['ID'].max()

np.int64(386)

In [39]:
df_RRHH['ID'].unique()

array([ 14,  36,   9,  28,  11,  13,  34,  22,  26,  20,  10,  15,  17,
        24,   3,  18,   7,   1,  30,   5,   6,   2,  31,  27,  33,  23,
        21,  25,  12,  32,  16,  29,  19,   8,   4,  35,  37,  38,  39,
        40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,
        53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,  65,
        66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,
        79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,
        92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103, 104,
       105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117,
       118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130,
       131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143,
       144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156,
       157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169,
       170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 18

In [40]:
# IDs con registros en cada entrega y empleados que aparecen por primera vez.
resumen_semanal = (
    df_RRHH
    .groupby('importacion', as_index=False)
    .agg(
        registros=('ID', 'size'),
        IDs_con_registros=('ID', 'nunique')
    )
)

primera_importacion_por_ID = (
    df_RRHH
    .groupby('ID', as_index=False)['importacion']
    .min()
)

ids_nuevos_por_fecha = (
    primera_importacion_por_ID
    .groupby('importacion')
    .size()
    .rename('IDs_nuevos')
)

resumen_semanal = (
    resumen_semanal
    .merge(ids_nuevos_por_fecha, on='importacion', how='left')
    .sort_values('importacion')
)

resumen_semanal


,importacion,registros,IDs_con_registros,IDs_nuevos
0,2026-05-25,739,36,36
1,2026-06-01,105,104,100
2,2026-06-08,265,260,250


## Reason absence 0 to nan - It is not a code and mainly corresponding to disciplinary failure 1

In [41]:
df_RRHH[df_RRHH['Reason_absence'] == 0]

,ID,Reason_absence,Month_absence,Day_week,Seasons,Transportation_expense,Distance_Residence_Work,Service_time,Age,Work_load_Average_day,...,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,BMI_calculated,Absenteeism_hours,importacion
696,36,0,7.0,3,1,118,13,18,50,239.554,...,1,1,0,0,98,178,31,30.9,0,2026-05-25
697,20,0,9.0,2,4,260,50,11,36,241.476,...,4,1,0,0,65,168,23,23.0,0,2026-05-25
699,11,0,9.0,3,4,289,36,13,33,241.476,...,2,1,0,1,90,172,30,30.4,0,2026-05-25
700,36,0,9.0,3,4,118,13,18,50,241.476,...,1,1,0,0,98,178,31,30.9,0,2026-05-25
701,13,0,9.0,4,4,369,17,12,31,241.476,...,3,1,0,0,70,169,25,24.5,0,2026-05-25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1037,329,0,10.0,4,4,118,13,18,50,253.465,...,1,1,0,0,98,178,31,30.9,0,2026-06-08
1045,337,0,10.0,3,4,157,27,6,29,265.017,...,0,1,1,0,75,185,22,21.9,0,2026-06-08
1054,346,0,9.0,3,4,118,13,18,50,241.476,...,1,1,0,0,98,178,31,30.9,0,2026-06-08
1058,350,0,6.0,2,3,235,29,12,48,275.089,...,1,0,1,5,88,163,33,33.1,0,2026-06-08


Change

In [42]:
df_RRHH['Reason_absence'] = df_RRHH['Reason_absence'].replace(0, np.nan)

Check change correct

In [43]:
df_RRHH['Reason_absence'].unique()

array([11., 13.,  6.,  9., 12., 19., 18.,  1., 10., 14.,  7., 28.,  2.,
       26., 23., 22., 21., 24., 17.,  8.,  5., 15.,  4., 25.,  3., 27.,
       16., nan])

In [44]:
df_RRHH.info()

<class 'pandas.DataFrame'>
Index: 1109 entries, 0 to 1109
Data columns (total 23 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   ID                       1109 non-null   int64         
 1   Reason_absence           1042 non-null   float64       
 2   Month_absence            1106 non-null   float64       
 3   Day_week                 1109 non-null   int64         
 4   Seasons                  1109 non-null   int64         
 5   Transportation_expense   1109 non-null   int64         
 6   Distance_Residence_Work  1109 non-null   int64         
 7   Service_time             1109 non-null   int64         
 8   Age                      1109 non-null   int64         
 9   Work_load_Average_day    1109 non-null   float64       
 10  Hit_target               1109 non-null   int64         
 11  Disciplinary_failure     1109 non-null   int64         
 12  Education                1109 non-null   int64    

## Seasons to correct season by month

Seasons (summer (1), autumn (2), winter (3), spring (4))  
España:  
- 1 (winter): 6 - 9 (originally 1 - not change) 
- 2 (spring): 9 - 12 (originally 4 - CHANGE) 
- 3 (summer): 12 - 3 (originally 2 - CHANGE)  
- 4 (autumn): 3 - 6 (originally 3 - CHANGE) 

Brasil se encuentra en el hemisferio sur y sus cuatro estaciones siguen el calendario austral:
- Primavera: del 22 de septiembre al 20 de diciembre.
- Verano: del 21 de diciembre al 19 de marzo (temporada alta y de lluvias en muchas zonas). 
- Otoño: del 20 de marzo al 20 de junio.
- Invierno: del 21 de junio al 21 de septiembre.

In [45]:
df_RRHH.groupby('Seasons')['Month_absence'].unique()

Seasons
1     [7.0, 8.0, 9.0, 6.0, nan]
2    [3.0, 12.0, 1.0, 2.0, nan]
3     [4.0, 3.0, 6.0, 5.0, nan]
4       [11.0, 12.0, 10.0, 9.0]
Name: Month_absence, dtype: object

Change

In [46]:
#Ordenamos los meses correctamente a Estacionalidad Brasil
df_RRHH['Seasons'] = df_RRHH['Seasons'].replace({
    4.0: 2.0,
    2.0: 3.0,
    3.0: 4.0
})

Check

In [47]:
df_RRHH.groupby('Seasons')['Month_absence'].unique()

Seasons
1     [7.0, 8.0, 9.0, 6.0, nan]
2       [11.0, 12.0, 10.0, 9.0]
3    [3.0, 12.0, 1.0, 2.0, nan]
4     [4.0, 3.0, 6.0, 5.0, nan]
Name: Month_absence, dtype: object

### Cambiar los meses para que las estaciones Brasileñas coincidad con las Españolas

Desplazamos 6 meses, es un cambio de hemisferio norte al sur

- Del 6 - 9 (Invierno Brasil) cambio a 12 - 3 (Invierno España)
- Del 9 - 12 (Primavera Brasil) cambio a 3 - 6 (Primavera España)
- Del 12 - 3 (Verano Brasil) cambio a 6 - 9 (Verano España)
- Del 3 - 6 (Otoño Brasil) cambio a 9 - 12 (Otoño españa)

In [48]:
mask = df_RRHH['Month_absence'].notna() #Cromprovacion de que el mas existe

df_RRHH.loc[mask, 'Month_absence'] = (
    (df_RRHH.loc[mask, 'Month_absence'] + 5) % 12
) + 1

In [49]:
df_RRHH.groupby('Seasons')['Month_absence'].unique()

Seasons
1      [1.0, 2.0, 3.0, 12.0, nan]
2            [5.0, 6.0, 4.0, 3.0]
3       [9.0, 6.0, 7.0, 8.0, nan]
4    [10.0, 9.0, 12.0, 11.0, nan]
Name: Month_absence, dtype: object

Se realizó un desplazamiento de seis meses en la variable Month_absence con el fin de adaptar la temporalidad del dataset brasileño al contexto español. De este modo, los patrones estacionales observados en variables como Hit_target o el absentismo conservan su interpretación climática y temporal, permitiendo que fenómenos asociados al verano, invierno o periodos vacacionales se mantengan coherentes en la simulación del contexto español.

## Create a column with the labels corresponding to the numbers

Dictionaries

In [50]:
reason_labels = {
    1: "Enfermedades infecciosas y parasitarias",
    2: "Neoplasias",
    3: "Sangre e inmunidad",
    4: "Endocrinas, nutricionales y metabólicas",
    5: "Trastornos mentales y del comportamiento",
    6: "Sistema nervioso",
    7: "Ojo y anexos",
    8: "Oído y apófisis mastoides",
    9: "Sistema circulatorio",
    10: "Sistema respiratorio",
    11: "Sistema digestivo",
    12: "Piel y tejido subcutáneo",
    13: "Sistema musculoesquelético",
    14: "Sistema genitourinario",
    15: "Embarazo, parto y puerperio",
    16: "Afecciones perinatales",
    17: "Malformaciones congénitas",
    18: "Síntomas y hallazgos no clasificados",
    19: "Lesiones, intoxicaciones y consecuencias externas",
    20: "Causas externas de morbilidad y mortalidad",
    21: "Factores de salud y contacto sanitario",
    22: "Seguimiento de paciente",
    23: "Consulta médica",
    24: "Donación de sangre",
    25: "Examen de laboratorio",
    26: "Ausencia injustificada",
    27: "Fisioterapia",
    28: "Consulta dental",
}

month_labels = {
    1.0: "Enero", 2.0: "Febrero", 3.0: "Marzo", 4.0: "Abril", 5.0: "Mayo", 6.0: "Junio",
    7.0: "Julio", 8.0: "Agosto", 9.0: "Septiembre", 10.0: "Octubre", 11.0: "Noviembre", 12.0: "Diciembre",
}

day_labels = {2: "Lunes", 3: "Martes", 4: "Miércoles", 5: "Jueves", 6: "Viernes"}
season_labels = {1: "Invierno", 2: "Primavera", 3: "Verano", 4: "Otoño"}
education_labels = {1: "Secundaria", 2: "Grado", 3: "Posgrado", 4: "Máster/Doctorado"}

Create function to do it for each column

In [51]:
def insert_column_labels(df, column: str, name_new_column: str, labels: dict):
    
    col_index = df.columns.get_loc(column)

    df.insert(
        col_index + 1,
        name_new_column,
        df[column].replace(labels)
    )

In [52]:
df_RRHH.columns

Index(['ID', 'Reason_absence', 'Month_absence', 'Day_week', 'Seasons',
       'Transportation_expense', 'Distance_Residence_Work', 'Service_time',
       'Age', 'Work_load_Average_day', 'Hit_target', 'Disciplinary_failure',
       'Education', 'Son', 'Social_drinker', 'Social_smoker', 'Pet', 'Weight',
       'Height', 'Body_mass_index', 'BMI_calculated', 'Absenteeism_hours',
       'importacion'],
      dtype='str')

Change

In [53]:
insert_column_labels(df_RRHH,'Month_absence','Month_absence_name',month_labels)
insert_column_labels(df_RRHH,'Reason_absence','Reason_absence_name',reason_labels)
insert_column_labels(df_RRHH,'Day_week','Day_week_name',day_labels)
insert_column_labels(df_RRHH,'Seasons','Seasons_name',season_labels)
insert_column_labels(df_RRHH,'Education','Education_name',education_labels)

In [54]:
df_RRHH.columns

Index(['ID', 'Reason_absence', 'Reason_absence_name', 'Month_absence',
       'Month_absence_name', 'Day_week', 'Day_week_name', 'Seasons',
       'Seasons_name', 'Transportation_expense', 'Distance_Residence_Work',
       'Service_time', 'Age', 'Work_load_Average_day', 'Hit_target',
       'Disciplinary_failure', 'Education', 'Education_name', 'Son',
       'Social_drinker', 'Social_smoker', 'Pet', 'Weight', 'Height',
       'Body_mass_index', 'BMI_calculated', 'Absenteeism_hours',
       'importacion'],
      dtype='str')

In [55]:
df_RRHH.info()

<class 'pandas.DataFrame'>
Index: 1109 entries, 0 to 1109
Data columns (total 28 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   ID                       1109 non-null   int64         
 1   Reason_absence           1042 non-null   float64       
 2   Reason_absence_name      1042 non-null   object        
 3   Month_absence            1106 non-null   float64       
 4   Month_absence_name       1106 non-null   object        
 5   Day_week                 1109 non-null   int64         
 6   Day_week_name            1109 non-null   object        
 7   Seasons                  1109 non-null   int64         
 8   Seasons_name             1109 non-null   object        
 9   Transportation_expense   1109 non-null   int64         
 10  Distance_Residence_Work  1109 non-null   int64         
 11  Service_time             1109 non-null   int64         
 12  Age                      1109 non-null   int64    

In [56]:
df_RRHH.sample(5)

,ID,Reason_absence,Reason_absence_name,Month_absence,Month_absence_name,Day_week,Day_week_name,Seasons,Seasons_name,Transportation_expense,...,Son,Social_drinker,Social_smoker,Pet,Weight,Height,Body_mass_index,BMI_calculated,Absenteeism_hours,importacion
1037,329,NaN,NaN,4.0,Abril,4,Miércoles,2,Primavera,118,...,1,1,0,0,98,178,31,30.9,0,2026-06-08
541,28,23.0,Consulta médica,5.0,Mayo,6,Viernes,2,Primavera,225,...,1,0,0,2,69,169,24,24.2,2,2026-05-25
1039,331,1.0,Enfermedades infecciosas y parasitarias,1.0,Enero,3,Martes,1,Invierno,260,...,4,1,0,0,65,168,23,23.0,8,2026-06-08
408,34,28.0,Consulta dental,5.0,Mayo,4,Miércoles,2,Primavera,118,...,0,0,0,0,83,172,28,28.1,3,2026-05-25
515,24,28.0,Consulta dental,9.0,Septiembre,3,Martes,4,Otoño,246,...,0,1,0,0,67,170,23,23.2,2,2026-05-25


Los duplicados se conservan deliberadamente: sin una fecha o un identificador único de evento no es posible distinguir entre una duplicación técnica y dos ausencias reales con las mismas características.


## Outliers

In [57]:
def detectar_outliers_iqr(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    return df[(df[col] < lower) | (df[col] > upper)]

In [58]:
continuous_cols = [
    'Age',
    'Height',
    'Weight',
    'Body_mass_index',
    'Absenteeism_hours',
    'Transportation_expense',
    'Distance_Residence_Work',
    'Service_time',
    'Work_load_Average_day',
    'Hit_target'
]

In [59]:
resumen_outliers = []

for col in continuous_cols:
    resumen_outliers.append({
        'Variable': col,
        'Recuento_outliers': detectar_outliers_iqr(df_RRHH, col).shape[0],
        'Min': df_RRHH[col].min(),
        'Max': df_RRHH[col].max(),
        'Media': df_RRHH[col].mean().round(2),
        'Mediana': df_RRHH[col].median()
    })

resumen_outliers = (
    pd.DataFrame(resumen_outliers)
    .sort_values('Recuento_outliers', ascending=False)
)

resumen_outliers

,Variable,Recuento_outliers,Min,Max,Media,Mediana
1,Height,176,163.000,196.000,172.07,170.000
4,Absenteeism_hours,92,0.000,120.000,9.16,3.000
8,Work_load_Average_day,46,205.917,378.884,270.15,264.249
9,Hit_target,27,81.000,100.000,94.64,95.000
0,Age,19,27.000,58.000,36.65,37.000
7,Service_time,7,1.000,29.000,12.66,13.000
5,Transportation_expense,5,118.000,388.000,221.31,225.000
2,Weight,0,56.000,108.000,79.05,83.000
3,Body_mass_index,0,19.000,38.000,26.69,25.000
6,Distance_Residence_Work,0,5.000,52.000,29.51,26.000


# Controles finales y exportación


In [60]:
# Validaciones finales: la exportación se detiene si cambia algún supuesto básico.
raw_actual = pd.read_csv(DATA_RRHH_RAW_FILE)

assert len(df_RRHH) == len(raw_actual) - 1, (
    'El número final de registros no coincide con el raw menos el registro descartado.'
)
assert df_RRHH['ID'].nunique() == raw_actual['ID'].nunique(), (
    'Se ha perdido o añadido algún empleado durante la limpieza.'
)
assert df_RRHH.columns[-1] == 'importacion'
assert df_RRHH['importacion'].notna().all()
assert set(df_RRHH['importacion'].dt.strftime('%Y-%m-%d')) == {
    get_import_date(raw_file) for raw_file in DATA_RRHH_RAW_FILES
}

assert df_RRHH['Disciplinary_failure'].isin([0, 1]).all()
assert df_RRHH['Social_drinker'].isin([0, 1]).all()
assert df_RRHH['Social_smoker'].isin([0, 1]).all()
assert df_RRHH['Day_week'].isin([2, 3, 4, 5, 6]).all()
assert df_RRHH['Seasons'].isin([1, 2, 3, 4]).all()
assert df_RRHH['Education'].isin([1, 2, 3, 4]).all()
assert df_RRHH['Month_absence'].dropna().isin(range(1, 13)).all()
assert df_RRHH['Reason_absence'].dropna().isin(range(1, 29)).all()
assert (df_RRHH['Absenteeism_hours'] >= 0).all()

variables_estables_ID = [
    'Transportation_expense', 'Distance_Residence_Work', 'Service_time',
    'Age', 'Education', 'Son', 'Social_drinker', 'Social_smoker',
    'Pet', 'Weight', 'Height', 'Body_mass_index'
]
assert all(
    df_RRHH.groupby('ID')[columna].nunique(dropna=False).max() == 1
    for columna in variables_estables_ID
), 'Existen características personales inconsistentes dentro de algún ID.'

duplicados_sin_importacion = (
    df_RRHH.drop(columns='importacion').duplicated().sum()
)
print('Controles finales superados.')
print(f'Duplicados conservados deliberadamente: {duplicados_sin_importacion}')
display(resumen_semanal)


Controles finales superados.
Duplicados conservados deliberadamente: 54


,importacion,registros,IDs_con_registros,IDs_nuevos
0,2026-05-25,739,36,36
1,2026-06-01,105,104,100
2,2026-06-08,265,260,250


In [61]:
df_RRHH.to_csv(DATA_OUTPUT_FILE, encoding='utf-8', index=False)
print(f'Archivo exportado: {DATA_OUTPUT_FILE}')


Archivo exportado: ../../Data/clean/clean_data_08062026.csv
